# 01 — Preprocessing

Runs QC, filtering, normalization, HVG selection, and PCA on both datasets independently.

| Dataset | Cells | Genes | Role |
|---------|-------|-------|------|
| Bhaduri et al. 2020 | 242,350 | 16,774 | Organoids |
| Zhong et al. 2018 | 2,394 | 24,153 | Fetal PFC reference |

**Prerequisites:** `colab_00_data_download.ipynb` must have been run. Both h5ad files must be on Drive.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q scanpy

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor='white')

DRIVE_ROOT = '/content/drive/MyDrive/brain-organoid-trajectories'

PATHS = {
    'bhaduri_raw':       os.path.join(DRIVE_ROOT, 'data/processed/bhaduri_2020/bhaduri_2020_raw.h5ad'),
    'zhong_raw':         os.path.join(DRIVE_ROOT, 'data/processed/zhong_2018/zhong_2018_raw.h5ad'),
    'bhaduri_processed': os.path.join(DRIVE_ROOT, 'data/processed/bhaduri_2020/bhaduri_2020_preprocessed.h5ad'),
    'zhong_processed':   os.path.join(DRIVE_ROOT, 'data/processed/zhong_2018/zhong_2018_preprocessed.h5ad'),
}

print('Paths configured.')
for k, v in PATHS.items():
    print(f'  {k}: {v}')

## 2. Load Raw Data

In [ ]:
adata_organoid = sc.read_h5ad(PATHS['bhaduri_raw'])
adata_fetal    = sc.read_h5ad(PATHS['zhong_raw'])

print(f'Bhaduri 2020: {adata_organoid.shape[0]:,} cells x {adata_organoid.shape[1]:,} genes')
print(f'Zhong 2018:   {adata_fetal.shape[0]:,} cells x {adata_fetal.shape[1]:,} genes')

## 3. Clean Bhaduri — Remove Empty Barcode

The raw matrix has an empty string as the first barcode — an artifact of the file format. Remove it before QC.

In [ ]:
mask = adata_organoid.obs_names != ''
adata_organoid = adata_organoid[mask].copy()
print(f'After removing empty barcode: {adata_organoid.shape[0]:,} cells')

## 4. QC — Bhaduri 2020 (Organoids)

Calculate per-cell QC metrics, then plot distributions to choose filtering thresholds.

- `n_genes_by_counts` — genes detected per cell (low = empty droplet, high = doublet)
- `total_counts` — total UMI counts per cell
- `pct_counts_mt` — % mitochondrial counts (high = dying/stressed cell)

In [ ]:
# Flag mitochondrial genes (MT- prefix in human)
adata_organoid.var['mt'] = adata_organoid.var_names.str.startswith('MT-')
print(f'Mitochondrial genes found: {adata_organoid.var["mt"].sum()}')

sc.pp.calculate_qc_metrics(
    adata_organoid,
    qc_vars=['mt'],
    percent_top=None,
    log1p=False,
    inplace=True
)

print('QC metrics calculated.')
print(adata_organoid.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].describe().round(1))

In [ ]:
# Violin plots — look at distributions to decide thresholds
sc.pl.violin(
    adata_organoid,
    ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
    jitter=0.4,
    multi_panel=True,
    show=True
)

In [ ]:
# Scatter: total_counts vs n_genes (doublets appear as outliers top-right)
sc.pl.scatter(adata_organoid, x='total_counts', y='n_genes_by_counts', color='pct_counts_mt')

In [ ]:
# Histogram of n_genes — useful when violin/scatter don't show a clear cutoff
# Look for a valley or a point where the tail becomes clearly sparse
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(adata_organoid.obs['n_genes_by_counts'], bins=200, color='steelblue', edgecolor='none')
axes[0].set_xlabel('n_genes_by_counts')
axes[0].set_ylabel('number of cells')
axes[0].set_title('Bhaduri — gene count distribution')

axes[1].hist(adata_organoid.obs['pct_counts_mt'], bins=100, color='salmon', edgecolor='none')
axes[1].set_xlabel('pct_counts_mt')
axes[1].set_ylabel('number of cells')
axes[1].set_title('Bhaduri — MT% distribution')

plt.tight_layout()
plt.show()

In [ ]:
# --- Set thresholds based on plots above ---
MIN_GENES   = 200    # remove empty droplets
MAX_GENES   = 6000   # remove doublets — adjust after looking at distribution
MAX_MT_PCT  = 20     # remove dying cells
MIN_CELLS   = 3      # remove genes detected in fewer than 3 cells

n_before = adata_organoid.n_obs

sc.pp.filter_cells(adata_organoid, min_genes=MIN_GENES)
sc.pp.filter_cells(adata_organoid, max_genes=MAX_GENES)
adata_organoid = adata_organoid[adata_organoid.obs['pct_counts_mt'] < MAX_MT_PCT].copy()
sc.pp.filter_genes(adata_organoid, min_cells=MIN_CELLS)

n_after = adata_organoid.n_obs
print(f'Cells before: {n_before:,}')
print(f'Cells after:  {n_after:,}')
print(f'Removed:      {n_before - n_after:,} ({100 * (n_before - n_after) / n_before:.1f}%)')
print(f'Genes remaining: {adata_organoid.n_vars:,}')

## 5. QC — Zhong 2018 (Fetal PFC)

Same steps, but Zhong is a smaller, already-curated dataset — expect less aggressive filtering.

In [ ]:
adata_fetal.var['mt'] = adata_fetal.var_names.str.startswith('MT-')
print(f'Mitochondrial genes found: {adata_fetal.var["mt"].sum()}')

sc.pp.calculate_qc_metrics(
    adata_fetal,
    qc_vars=['mt'],
    percent_top=None,
    log1p=False,
    inplace=True
)

print('QC metrics calculated.')
print(adata_fetal.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].describe().round(1))

In [ ]:
sc.pl.violin(
    adata_fetal,
    ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
    jitter=0.4,
    multi_panel=True,
    show=True
)

In [ ]:
sc.pl.scatter(adata_fetal, x='total_counts', y='n_genes_by_counts', color='pct_counts_mt')

In [ ]:
# Histogram of n_genes — useful when violin/scatter don't show a clear cutoff
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(adata_fetal.obs['n_genes_by_counts'], bins=100, color='steelblue', edgecolor='none')
axes[0].set_xlabel('n_genes_by_counts')
axes[0].set_ylabel('number of cells')
axes[0].set_title('Zhong — gene count distribution')

axes[1].hist(adata_fetal.obs['pct_counts_mt'], bins=50, color='salmon', edgecolor='none')
axes[1].set_xlabel('pct_counts_mt')
axes[1].set_ylabel('number of cells')
axes[1].set_title('Zhong — MT% distribution')

plt.tight_layout()
plt.show()

In [ ]:
# --- Set thresholds based on plots above ---
MIN_GENES   = 200
MAX_GENES   = 8000   # Zhong cells may have higher gene counts — adjust after plots
MAX_MT_PCT  = 20
MIN_CELLS   = 3

n_before = adata_fetal.n_obs

sc.pp.filter_cells(adata_fetal, min_genes=MIN_GENES)
sc.pp.filter_cells(adata_fetal, max_genes=MAX_GENES)
adata_fetal = adata_fetal[adata_fetal.obs['pct_counts_mt'] < MAX_MT_PCT].copy()
sc.pp.filter_genes(adata_fetal, min_cells=MIN_CELLS)

n_after = adata_fetal.n_obs
print(f'Cells before: {n_before:,}')
print(f'Cells after:  {n_after:,}')
print(f'Removed:      {n_before - n_after:,} ({100 * (n_before - n_after) / n_before:.1f}%)')
print(f'Genes remaining: {adata_fetal.n_vars:,}')

## 6. Normalize and Log-Transform

Each cell has a different total count (sequencing depth) — deeper sequencing ≠ more gene expression.  
We normalize so every cell sums to 10,000 counts, then log-transform to compress the dynamic range.

After this step, `adata.X` contains log-normalized values. Raw counts are preserved in `adata.raw`.

In [ ]:
for name, adata in [('Bhaduri', adata_organoid), ('Zhong', adata_fetal)]:
    adata.raw = adata  # freeze raw counts before normalization
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    print(f'{name}: normalized and log-transformed.')

## 7. Highly Variable Gene Selection

We select the top 2,000 HVGs per dataset independently. These are genes with high variance across cells — they carry the most biological signal for clustering and trajectory analysis.

HVG selection is done **before** integration. At integration time we will take the intersection of HVGs shared between both datasets.

In [ ]:
for name, adata in [('Bhaduri', adata_organoid), ('Zhong', adata_fetal)]:
    sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat')
    n_hvg = adata.var['highly_variable'].sum()
    print(f'{name}: {n_hvg:,} HVGs selected')
    sc.pl.highly_variable_genes(adata, show=True)

## 8. PCA

Reduce dimensionality from ~2,000 HVGs to 50 principal components.  
PCA is run only on HVGs (`use_highly_variable=True`).

We use 50 PCs here — we'll check the variance explained (elbow plot) to decide how many to carry forward into UMAP.

In [ ]:
for name, adata in [('Bhaduri', adata_organoid), ('Zhong', adata_fetal)]:
    sc.pp.scale(adata, max_value=10)  # scale to unit variance, clip at 10
    sc.tl.pca(adata, n_comps=50, use_highly_variable=True)
    print(f'{name}: PCA done.')
    sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True, show=True)

## 9. Save Preprocessed Objects

In [ ]:
adata_organoid.write_h5ad(PATHS['bhaduri_processed'])
adata_fetal.write_h5ad(PATHS['zhong_processed'])

for k in ['bhaduri_processed', 'zhong_processed']:
    size_mb = os.path.getsize(PATHS[k]) / 1e6
    print(f'Saved: {PATHS[k]} ({size_mb:.1f} MB)')

## Done

Both datasets are preprocessed:
- QC filtered (empty droplets, doublets, dying cells removed)
- Normalized to 10k counts + log-transformed
- 2,000 HVGs selected per dataset
- PCA computed (50 PCs)
- Raw counts preserved in `adata.raw`

Saved to Drive:
- `data/processed/bhaduri_2020/bhaduri_2020_preprocessed.h5ad`
- `data/processed/zhong_2018/zhong_2018_preprocessed.h5ad`

Next: `colab_02_umap_clustering.ipynb` — neighbor graph, UMAP, Leiden clustering, cell type annotation.